# OmniBird — single-modality event JEPA (synthetic dataset)

End-to-end training of OmniBird-JEPA on **synthetic event clips** — no
EventScape download required. Once this works, swap the synthetic dataset
for `EventScapeDataset(root, mode="events_only")` to run on real CARLA
simulation data with the same code below.

**Architecture** (carried over from PointBigBird v2):
- Per-token Transformer encoder (signal_dim=1 polarity, coord_dim=3 (x,y,t))
- BigBird block-sparse self-attention, block_size=8
- Per-layer random serialization: 3-D Morton / Hilbert / reverses
- i-JEPA multi-block disjoint masking on the event cloud
- AttnPool probe head, cosine loss, EMA cap 0.9999, no DINO centering


## 1. Setup

In [ ]:
import os, sys, math, time, copy, ssl
sys.path.insert(0, os.path.abspath('..'))
ssl._create_default_https_context = ssl._create_unverified_context

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import matplotlib.pyplot as plt

from omnibird import (
    OmniBirdConfig, OmniBirdEncoder, OmniBirdPredictor,
    orderings_from_batch, TargetCenter, ema_update, make_momentum_schedule,
    gather_target_features, jepa_loss, diag_dict, fmt_diag,
    quick_probe, save_atomic, ensure_dir, short_params, count_params,
)
from datasets import build_synthetic_loaders, SyntheticEventDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)

cfg = OmniBirdConfig()
cfg.epochs         = 100
cfg.batch_size     = 32
cfg.lr             = 2e-4
cfg.probe_interval = 10
cfg.probe_epochs   = 3
cfg.probe_patience = 5
cfg.log_every      = 25
cfg.ckpt_dir       = "./checkpoints_omnibird_synth"
ensure_dir(cfg.ckpt_dir)

print(f"DEVICE = {DEVICE}")
print(f"events/sample = {cfg.n_events_total}  n_ctx = {cfg.n_ctx}  n_tgt = {cfg.n_tgt}")
print(f"D_model = {cfg.d_model}  block_size = {cfg.block_size}  n_layers = {cfg.n_layers_enc}")


## 2. Synthetic event dataset + loaders

In [ ]:
train_loader, train_eval_loader, test_loader = build_synthetic_loaders(
    cfg, n_train=5000, n_test=1000
)
print(f"train={len(train_loader.dataset)}  train_eval={len(train_eval_loader.dataset)}  test={len(test_loader.dataset)}")

# Peek at one batch
batch = next(iter(train_loader))
for k, v in batch.items():
    if hasattr(v, "shape"):
        print(f"  {k:24s} {tuple(v.shape)}  {v.dtype}")


## 3. Visualize one event clip

In [ ]:
ds_peek = SyntheticEventDataset(n_samples=10, n_events_per_sample=cfg.n_events_total, seed=42)
fig = plt.figure(figsize=(15, 5))
for i in range(3):
    ev, lab = ds_peek[i]
    ax = fig.add_subplot(1, 3, i+1, projection='3d')
    pos = ev[ev[:,3] > 0]; neg = ev[ev[:,3] < 0]
    ax.scatter(pos[:,0], pos[:,1], pos[:,2], s=1, c='red',  alpha=0.4, label='+1')
    ax.scatter(neg[:,0], neg[:,1], neg[:,2], s=1, c='blue', alpha=0.4, label='−1')
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('t')
    ax.set_title(f"class {lab}  ({ev.shape[0]} events)")
    if i == 0: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()


## 4. Models

In [ ]:
context_encoder = OmniBirdEncoder(
    d_model=cfg.d_model, n_layers=cfg.n_layers_enc,
    n_heads=cfg.n_heads, dim_head=cfg.dim_head,
    block_size=cfg.block_size, window=cfg.window,
    n_random=cfg.n_random, n_global=cfg.n_global,
    ffn_mult=cfg.ffn_mult,
    signal_dim=cfg.signal_dim, coord_dim=cfg.coord_dim,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    serial_orders=cfg.serial_orders,
    reinject_pos=cfg.reinject_pos,
).to(DEVICE)
target_encoder = copy.deepcopy(context_encoder).to(DEVICE)
for q in target_encoder.parameters(): q.requires_grad_(False)
predictor = OmniBirdPredictor(
    d_model=cfg.d_model, d_pred=cfg.d_pred,
    n_layers=cfg.n_layers_pred,
    n_heads=cfg.n_heads_pred, dim_head=cfg.dim_head_pred,
    fourier_dim=cfg.fourier_dim, fourier_scale=cfg.fourier_scale,
    ffn_mult=cfg.ffn_mult, pos_symmetric=cfg.predictor_pos_symmetric,
).to(DEVICE)
target_center = TargetCenter(cfg.d_model, momentum=0.9).to(DEVICE)
print(f"context_encoder: {short_params(context_encoder)}  predictor: {short_params(predictor)}")


## 5. Optimizer + schedules + state

In [ ]:
params = list(context_encoder.parameters()) + list(predictor.parameters())
optimizer = AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
total_steps = cfg.epochs * len(train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps)
momentum_gen = make_momentum_schedule(cfg.ema_start, cfg.ema_end, total_steps)

PBB_LAST       = os.path.join(cfg.ckpt_dir, "omnibird_last.pt")
PBB_BEST       = os.path.join(cfg.ckpt_dir, "omnibird_best.pt")
PBB_PROBE_BEST = os.path.join(cfg.ckpt_dir, "omnibird_probe_best.pt")

RESUME = True
history = {"loss": [], "diag_steps": [], "diag_log": [], "probe_accs": []}
start_epoch, best_loss, global_step = 1, float("inf"), 0
best_probe_acc, probe_no_improve = -1.0, 0
m = cfg.ema_start

if RESUME and os.path.exists(PBB_LAST):
    s = torch.load(PBB_LAST, map_location=DEVICE, weights_only=False)
    context_encoder.load_state_dict(s["context_encoder"])
    target_encoder.load_state_dict(s["target_encoder"])
    predictor.load_state_dict(s["predictor"])
    target_center.load_state_dict(s["center"])
    optimizer.load_state_dict(s["opt"]); scheduler.load_state_dict(s["sched"])
    history = s.get("history", history)
    global_step = s.get("global_step", 0)
    best_loss = s.get("best_loss", float("inf"))
    best_probe_acc = s.get("best_probe_acc", -1.0)
    probe_no_improve = s.get("probe_no_improve", 0)
    start_epoch = s["epoch"] + 1
    for _ in range(global_step):
        try: m = next(momentum_gen)
        except StopIteration: m = cfg.ema_end
    print(f"resumed @ ep {s['epoch']}, step {global_step}, best_probe_acc={best_probe_acc:.4f}")
else:
    print("starting fresh.")


## 6. Embedded probe (uses train_eval_loader + test_loader)

In [ ]:
def move_orderings(ords, device):
    return {k: {kk: vv.to(device) for kk, vv in v.items()} for k, v in ords.items()}

def probe_now(num_epochs=cfg.probe_epochs, lr=1e-3):
    return quick_probe(
        context_encoder, train_eval_loader, test_loader,
        d_model=cfg.d_model, n_classes=10,
        num_epochs=num_epochs, lr=lr, weight_decay=1e-4, device=DEVICE,
        use_attn_pool=cfg.probe_use_attn_pool,
    )


## 7. Training loop

In [ ]:
epoch = start_epoch - 1
try:
    for epoch in range(start_epoch, cfg.epochs + 1):
        context_encoder.train(); predictor.train()
        epoch_loss, steps = 0.0, 0
        t0 = time.time()

        for batch in train_loader:
            ctx_s  = batch["ctx_signal"].to(DEVICE)
            ctx_c  = batch["ctx_coords"].to(DEVICE)
            pool_s = batch["pool_signal"].to(DEVICE)
            pool_c = batch["pool_coords"].to(DEVICE)
            tgt_c  = batch["tgt_coords"].to(DEVICE)
            tgt_pp = batch["tgt_pool_pos"].to(DEVICE)
            ctx_o  = move_orderings(orderings_from_batch(batch, "ctx"),  DEVICE)
            pool_o = move_orderings(orderings_from_batch(batch, "pool"), DEVICE)

            with torch.no_grad():
                g_tgt = target_encoder(pool_s, pool_c, pool_o)
                h_tgt_raw = gather_target_features(g_tgt, tgt_pp)
                if cfg.use_centering:
                    target_center.update(h_tgt_raw)
                    h_tgt = F.layer_norm(target_center(h_tgt_raw), (h_tgt_raw.size(-1),))
                else:
                    h_tgt = F.layer_norm(h_tgt_raw, (h_tgt_raw.size(-1),))

            g_ctx  = context_encoder(ctx_s, ctx_c, ctx_o)
            h_pred = predictor(g_ctx, tgt_c, ctx_coords=ctx_c if cfg.predictor_pos_symmetric else None)

            loss = jepa_loss(h_pred, h_tgt, loss_type=cfg.loss_type)
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step(); scheduler.step()

            try: m = next(momentum_gen)
            except StopIteration: m = cfg.ema_end
            ema_update(target_encoder, context_encoder, m)

            global_step += 1; epoch_loss += loss.item(); steps += 1
            if global_step % cfg.log_every == 0:
                d = diag_dict(loss, h_pred, h_tgt, g_ctx, target_center)
                print(fmt_diag(d, global_step, epoch, scheduler.get_last_lr()[0], m))
                history["diag_steps"].append(global_step); history["diag_log"].append(d)

        avg = epoch_loss / max(steps, 1)
        history["loss"].append(avg)
        improved = avg < best_loss
        if improved: best_loss = avg

        ckpt_state = {
            "epoch": epoch, "cfg": cfg.__dict__,
            "context_encoder": context_encoder.state_dict(),
            "target_encoder":  target_encoder.state_dict(),
            "predictor":       predictor.state_dict(),
            "center":          target_center.state_dict(),
            "opt": optimizer.state_dict(), "sched": scheduler.state_dict(),
            "global_step": global_step, "best_loss": best_loss,
            "best_probe_acc": best_probe_acc, "probe_no_improve": probe_no_improve,
            "history": history,
        }
        if improved: save_atomic(ckpt_state, PBB_BEST)
        save_atomic(ckpt_state, PBB_LAST)
        print(f"=== ep {epoch:03d}/{cfg.epochs}  avg_loss={avg:.4f}  m={m:.4f}  {time.time()-t0:.1f}s"
              + ("  ★ new best (saved omnibird_best.pt)" if improved else ""))

        # Embedded probe + early stop
        if cfg.probe_interval > 0 and epoch % cfg.probe_interval == 0:
            t_probe = time.time()
            print(f"  → Running {cfg.probe_epochs}-epoch linear probe at epoch {epoch}...")
            acc = probe_now()
            history.setdefault("probe_accs", []).append((epoch, acc))
            print(f"  [probe ep {epoch:03d}]  test_acc = {acc:.4f}  ({time.time()-t_probe:.1f}s)")
            if acc > best_probe_acc:
                best_probe_acc = acc; probe_no_improve = 0
                ckpt_state["best_probe_acc"]   = best_probe_acc
                ckpt_state["probe_no_improve"] = probe_no_improve
                ckpt_state["history"]          = history
                save_atomic(ckpt_state, PBB_PROBE_BEST)
                print(f"  ★ new best probe acc → {best_probe_acc:.4f}  (saved omnibird_probe_best.pt)")
            else:
                probe_no_improve += 1
                pp = cfg.probe_patience if cfg.probe_patience > 0 else "∞"
                print(f"  no probe improvement ({probe_no_improve}/{pp})  best so far = {best_probe_acc:.4f}")
            if cfg.probe_patience > 0 and probe_no_improve >= cfg.probe_patience:
                print(f"\nEarly stopping at epoch {epoch}: best probe acc = {best_probe_acc:.4f}")
                break

    print("\nDone.")
except KeyboardInterrupt:
    print(f"\nInterrupted at epoch {epoch}.  Checkpoints in {cfg.ckpt_dir}.")


## 8. Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
if history["loss"]:
    axes[0].plot(range(1, len(history["loss"])+1), history["loss"], lw=1.5)
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("avg loss"); axes[0].set_title("JEPA loss"); axes[0].grid(alpha=0.3)
if history["diag_log"]:
    steps = history["diag_steps"]
    cos_mean = [d["cos_mean"] for d in history["diag_log"]]
    axes[1].plot(steps, cos_mean, color='C2'); axes[1].set_ylim(-0.1, 1.05)
    axes[1].set_xlabel("step"); axes[1].set_title("cos(h_pred, h_tgt)"); axes[1].grid(alpha=0.3)
if history.get("probe_accs"):
    eps, accs = zip(*history["probe_accs"])
    axes[2].plot(eps, [a*100 for a in accs], 'o-', color='C3', markersize=6)
    axes[2].set_ylim(0, 100)
    axes[2].set_xlabel("epoch"); axes[2].set_ylabel("probe acc (%)")
    axes[2].set_title(f"probe acc (best = {max(accs)*100:.2f}%)"); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()
